# 4. Autonomous AI Agents & Multi-Agent Orchestration

This notebook explores autonomous agent architectures:
1. **ReAct Paradigm (Reasoning + Acting)**
2. **Tool Routing & Dynamic Execution Engine**
3. **Short-Term Context & Long-Term Memory Systems**
4. **Multi-Agent Orchestration Framework** (Planner, Researcher, Evaluator)

In [ ]:
import json
import re
from typing import List, Dict, Tuple, Any, Callable

print("=== Autonomous Agents Setup Complete ===")

## Step 1: Tool Registry & Execution Suite

Agents expand LLM capabilities by executing external functions: databases, APIs, calculators, and web searches.

In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Tuple[Callable, str]] = {}

    def register(self, name: str, description: str):
        def decorator(func: Callable):
            self.tools[name] = (func, description)
            return func
        return decorator

    def execute(self, name: str, **kwargs) -> str:
        if name not in self.tools:
            return f"Error: Tool '{name}' not found."
        func, _ = self.tools[name]
        try:
            return str(func(**kwargs))
        except Exception as e:
            return f"Tool Execution Error: {str(e)}"

registry = ToolRegistry()

@registry.register("calculator", "Evaluate mathematical expressions. Input: expr (str)")
def calculator(expr: str) -> float:
    # Safe evaluation of basic math
    allowed = set("0123456789+-*/. ()")
    if not all(c in allowed for c in expr):
        raise ValueError("Invalid characters in math expression")
    return eval(expr)

@registry.register("kb_search", "Search internal company knowledge base. Input: query (str)")
def kb_search(query: str) -> str:
    kb_db = {
        "vacation policy": "Employees receive 30 days of paid annual leave.",
        "wifi password": "Guest WiFi password is 'LuxcaraGuest2026!'.",
        "support email": "Contact IT at support@example.com."
    }
    for k, v in kb_db.items():
        if k in query.lower():
            return v
    return "No matching knowledge base entry found."

print("Registered Tools:", list(registry.tools.keys()))
print("Test Tool Execution (calculator 15*4+10):", registry.execute("calculator", expr="15*4+10"))
print("Test Tool Execution (kb_search vacation):", registry.execute("kb_search", query="vacation policy"))

## Step 2: The ReAct Execution Loop

The **ReAct** (Reason + Act) loop iteratively alternates between:
- **Thought**: Reasoning about current state and determining next step.
- **Action**: Selecting a tool and producing input parameters.
- **Observation**: Receiving tool execution results and feeding them back to Thought.
- **Final Answer**: Synthesizing answer when goal is met.

In [ ]:
class ReActAgent:
    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry

    def run(self, goal: str, max_iterations: int = 4) -> Dict[str, Any]:
        history = [f"Goal: {goal}"]
        print(f"--- Starting ReAct Execution for Goal: '{goal}' ---")

        # Simulated ReAct iteration sequence
        simulated_steps = [
            {
                "thought": "I need to look up the company vacation policy in the knowledge base.",
                "action": "kb_search",
                "action_input": {"query": "vacation policy"}
            },
            {
                "thought": "I found the annual leave count (30 days). Now I will compute remaining days if 12 are used.",
                "action": "calculator",
                "action_input": {"expr": "30 - 12"}
            }
        ]

        for step_idx, step in enumerate(simulated_steps, 1):
            thought = step["thought"]
            action = step["action"]
            action_input = step["action_input"]

            print(f"\nIteration {step_idx}:")
            print(f" Thought: {thought}")
            print(f" Action: {action}({action_input})")

            # Execute action
            obs = self.registry.execute(action, **action_input)
            print(f" Observation: {obs}")

            history.append(f"Thought: {thought}")
            history.append(f"Action: {action}({action_input})")
            history.append(f"Observation: {obs}")

        final_answer = "Employees have 30 days of vacation per year. After taking 12 days, 18 days remain."
        print(f"\nFinal Answer: {final_answer}")
        
        return {
            "goal": goal,
            "history": history,
            "final_answer": final_answer
        }

agent = ReActAgent(registry)
res = agent.run("How many annual leave days remain if an employee takes 12 days off?")

## Step 3: Short-Term & Long-Term Memory Systems

- **Short-Term Memory**: Conversation window tracking sliding history context.
- **Long-Term Memory**: Key-Value / Semantic Vector buffer storing persistent facts across multi-turn sessions.

In [ ]:
class AgentMemory:
    def __init__(self, max_short_term: int = 5):
        self.short_term: List[Dict[str, str]] = []
        self.long_term: Dict[str, str] = {}
        self.max_short_term = max_short_term

    def add_short_term(self, role: str, content: str):
        self.short_term.append({"role": role, "content": content})
        if len(self.short_term) > self.max_short_term:
            self.short_term.pop(0) # eviction

    def store_long_term(self, key: str, value: str):
        self.long_term[key] = value

    def recall_long_term(self, key: str) -> str:
        return self.long_term.get(key, "Fact not remembered.")

mem = AgentMemory(max_short_term=3)
mem.store_long_term("user_preferred_language", "German")
mem.add_short_term("user", "Hello!")
mem.add_short_term("assistant", "Hi there!")
mem.add_short_term("user", "What is my language setting?")

print("Recalled Long-Term Fact:", mem.recall_long_term("user_preferred_language"))
print("Current Short-Term Window Count:", len(mem.short_term))

## Step 4: Multi-Agent Orchestration

Multi-agent coordination breaks complex pipelines into specialised roles:
1. **Planner Agent**: Decomposes user request into ordered sub-tasks.
2. **Specialist Agents**: Execute domain-specific actions (e.g. Research, Code, QA).
3. **Evaluator Agent**: Validates quality and approves final output.

In [ ]:
class MultiAgentSystem:
    def plan(self, objective: str) -> List[str]:
        return [
            "Sub-task 1: Gather benchmark metric requirements.",
            "Sub-task 2: Synthesize performance summary.",
            "Sub-task 3: Review summary for technical accuracy."
        ]

    def execute_plan(self, tasks: List[str]) -> Dict[str, str]:
        outputs = {}
        # Specialist 1: Research Agent
        outputs["Task 1"] = "Collected metrics: p99 latency < 50ms, throughput = 2000 RPS."
        
        # Specialist 2: Writer Agent
        outputs["Task 2"] = f"Drafted Report: System achieves high throughput (2000 RPS) with low latency ({outputs['Task 1']})."
        
        # Specialist 3: QA Reviewer Agent
        outputs["Task 3"] = "PASSED: Report meets all quality standards."
        return outputs

mas = MultiAgentSystem()
tasks = mas.plan("Create performance benchmark report")
print("Orchestrated Plan Steps:", tasks)
results = mas.execute_plan(tasks)
print("\nMulti-Agent Pipeline Output:")
for task, res in results.items():
    print(f" {task}: {res}")